# 01. Первый агент: state, tools и filesystem

## Что агент пока не умеет

После `00` у нас есть идея agent harness, но ещё нет агента, который сам исследует рабочую директорию.

> Модель умеет рассуждать о коде только тогда, когда человек принёс код в контекст. Агент сам исследует окружение и решает, какие данные ему нужны.


## Какую способность добавляем

Сначала создаём минимальный Deep Agent без filesystem. Затем добавляем `FilesystemBackend` и показываем контраст до/после.

```text
Model  → решает, что делать дальше
State  → хранит сообщения и результаты действий
Tools  → позволяют воздействовать на окружение
```


## Workspace boundary

В нашем demo workspace — корень репозитория. `virtual_mode=True` нормализует пути внутри backend, но это не полноценная OS-песочница.

```text
Физический путь: локальный каталог на компьютере
Виртуальный путь: рабочее пространство агента внутри backend
```

Shell выключен по умолчанию. Если включить `OPENCLAW_ENABLE_LOCAL_SHELL=1`, shell не наследует env процесса (`inherit_env=False`).


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'workshop_notebooks' / 'openclaw_path'):
    if (candidate / 'workshop_utils.py').exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from workshop_utils import REPO_ROOT, model_name, print_stage_context, register_graphs, workspace_root, write_text

print_stage_context()
print("pyproject:", REPO_ROOT / "pyproject.toml")


In [ ]:
from deepagents import create_deep_agent

BASE_PROMPT = """\
You are OpenClaw at stage 01. Respond in Russian by default.
Explain honestly whether you can inspect repository files.
"""

minimal_agent = create_deep_agent(model=model_name(), tools=[], system_prompt=BASE_PROMPT)
print(type(minimal_agent).__name__)


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\n\nload_dotenv()\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\ndef _workspace_root() -> Path:\n    return Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).expanduser().resolve()\n\n\ndef _backend(*, require_shell: bool = False):\n    root = _workspace_root()\n    shell_enabled = os.getenv(\"OPENCLAW_ENABLE_LOCAL_SHELL\") == \"1\"\n    if require_shell and not shell_enabled:\n        raise RuntimeError(\"OPENCLAW_ENABLE_LOCAL_SHELL=1 is required for this stage\")\n    if shell_enabled:\n        from deepagents.backends import LocalShellBackend\n\n        return LocalShellBackend(\n            root_dir=root,\n            virtual_mode=True,\n            inherit_env=False,\n            timeout=120,\n            max_output_bytes=80_000,\n        )\n\n    from deepagents.backends import FilesystemBackend\n\n    return FilesystemBackend(root_dir=root, virtual_mode=True)\n\nBASE_PROMPT = \"\"\"\\\nYou are OpenClaw at stage 01. Respond in the user's language; default to Russian.\nExplain your limits honestly. Before filesystem is enabled, you cannot inspect the repository.\nWith filesystem backend, cite files for repository claims.\n\"\"\"\nagent = create_deep_agent(model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL), tools=[], system_prompt=BASE_PROMPT)\nagent_with_filesystem = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=[],\n    system_prompt=BASE_PROMPT,\n    backend=_backend(),\n)\n"
print(write_text("agents/openclaw_01_agent_and_filesystem.py", ENTRYPOINT).relative_to(REPO_ROOT))
print(register_graphs({
    "openclaw_01": "./agents/openclaw_01_agent_and_filesystem.py:agent",
    "openclaw_01_fs": "./agents/openclaw_01_agent_and_filesystem.py:agent_with_filesystem",
}).relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
Найди файл pyproject.toml и назови имя проекта и три зависимости. Если у тебя нет доступа к файлам, прямо скажи об этом.
```

### Ожидаемое поведение

1. `openclaw_01` честно говорит, что не имеет файлового доступа.
2. `openclaw_01_fs` читает `pyproject.toml` и цитирует вывод из файла.

### Что изменилось относительно предыдущего этапа

Появился контраст между моделью без среды и агентом с workspace.

### Текущее ограничение

Агент умеет исследовать локальную среду, но пока не взаимодействует с системами за пределами репозитория.
